# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 9:** 25 more useful B1 verbs (none repeated from Lesson 8). For each verb: English meaning → repeat the infinitive 3× (2nd and 3rd slow) → 4 sentences rotating tenses (presente, passato prossimo, imperfetto, futuro, condizionale, congiuntivo, imperativo, passive) and persons (io, tu, lui/lei, noi, voi, loro, Lei). Warm-up with Lesson 8 verbs, review rounds after verbs 8 and 16, full review at the end. False friends flagged: *pretendere*, *sopportare*, *realizzare*.

| # | verb | meaning |
|---|---|---|
| 1 | *accettare* | To accept |
| 2 | *rifiutare* | To refuse, to turn down |
| 3 | *ammettere* | To admit |
| 4 | *nascondere* | To hide |
| 5 | *scoprire* | To discover, to find out |
| 6 | *raccontare* | To tell, a story or an event |
| 7 | *ringraziare* | To thank |
| 8 | *scusarsi* | To apologize |
| 9 | *dimostrare* | To show, to prove |
| 10 | *tradurre* | To translate |
| 11 | *riconoscere* | To recognize, or to acknowledge |
| 12 | *confrontare* | To compare |
| 13 | *pretendere* | To demand, to expect too much |
| 14 | *sopportare* | To stand, to put up with |
| 15 | *realizzare* | To make real, to carry out, to build |
| 16 | *aggiungere* | To add |
| 17 | *togliere* | To take off, to remove |
| 18 | *accendere* | To turn on, to light |
| 19 | *spostare* | To move something, or to reschedule |
| 20 | *rimandare* | To postpone, to put off |
| 21 | *riempire* | To fill |
| 22 | *svolgere* | To carry out, to do a task |
| 23 | *tenere* | To hold, to keep |
| 24 | *coinvolgere* | To involve |
| 25 | *insistere* | To insist |

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson09_25Verbi_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 9 (B1): 25 more useful verbs
# Each verb: meaning -> repeat x3 -> 4 sentences across tenses & persons.
# "you" = tu unless the prompt says formal (Lei) or plural (voi).
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson nine. Twenty-five more useful verbs.
(narrator) As in lesson eight, you will hear the meaning. Repeat the verb three times. Then make sentences in different tenses, with different people.
(narrator) When the prompt says "you", use {tu}. When it says "formal", use {Lei}. When it says "you, plural", use {voi}.
(narrator) First, a warm-up with verbs from lesson eight.
(prompt) Say: I can't manage to sleep. || Non riesco a dormire.
(prompt) Say to a friend: I hope you are well. || Spero che tu stia bene.
(prompt) Say: I'll take care of it. || Me ne occupo io.
(prompt) Say: if I won the lottery, I would travel the world. || Se vincessi la lotteria, girerei il mondo.
(narrator) Now, new verbs.

(section) 1 — accettare
(narrator) Verb one. To accept: {accettare}.
(repeat x3 slow) accettare
(prompt) Say to a friend: I accept your apology. || Accetto le tue scuse.
(prompt) Say: she accepted the job offer. || Ha accettato l'offerta di lavoro.
(prompt) Ask the staff in a shop: do you accept credit cards? Use {voi}. || Accettate carte di credito?
(prompt) Say: I hope they accept our proposal. || Spero che accettino la nostra proposta.

(section) 2 — rifiutare
(narrator) Verb two. To refuse, to turn down: {rifiutare}.
(repeat x3 slow) rifiutare
(prompt) Say: he turned down the invitation. || Ha rifiutato l'invito.
(prompt) Ask a friend: why do you refuse to help me? || Perché rifiuti di aiutarmi?
(prompt) Say: we will refuse. || Rifiuteremo.
(prompt) Say: if they offered it to me, I would refuse. || Se me lo offrissero, rifiuterei.

(section) 3 — ammettere
(narrator) Verb three. To admit: {ammettere}. The past participle is {ammesso}. It also means: to admit someone, to a school.
(repeat x3 slow) ammettere
(prompt) Say: I admit I was wrong. || Ammetto di aver sbagliato.
(prompt) Say: he never admits his mistakes. || Non ammette mai i suoi errori.
(prompt) Tell a friend: admit it! || Ammettilo!
(prompt) Say: they were admitted to university. || Sono stati ammessi all'università.

(section) 4 — nascondere
(narrator) Verb four. To hide: {nascondere}. The past participle is {nascosto}.
(repeat x3 slow) nascondere
(prompt) Ask a friend: where did you hide the keys? || Dove hai nascosto le chiavi?
(prompt) Say: I have nothing to hide. || Non ho niente da nascondere.
(prompt) Say: the children hid in the garden. || I bambini si sono nascosti in giardino.
(prompt) Say: I think she is hiding something. || Credo che nasconda qualcosa.

(section) 5 — scoprire
(narrator) Verb five. To discover, to find out: {scoprire}. The past participle is {scoperto}.
(repeat x3 slow) scoprire
(prompt) Say: we discovered a wonderful restaurant. || Abbiamo scoperto un ristorante meraviglioso.
(prompt) Say: I found out the truth. || Ho scoperto la verità.
(prompt) Say: if they find out, they will get angry. || Se lo scoprono, si arrabbieranno.
(prompt) Say to a friend: one day you will find out that I was right. || Un giorno scoprirai che avevo ragione.

(section) 6 — raccontare
(narrator) Verb six. To tell, a story or an event: {raccontare}.
(repeat x3 slow) raccontare
(prompt) Tell a friend: tell me everything! || Raccontami tutto!
(prompt) Say: my grandfather used to tell a lot of stories. || Mio nonno raccontava molte storie.
(prompt) Say to two friends: I'll tell you about it tomorrow. || Ve lo racconto domani.
(prompt) Ask a friend: what did she tell you? || Che cosa ti ha raccontato?

(section) 7 — ringraziare
(narrator) Verb seven. To thank: {ringraziare}.
(repeat x3 slow) ringraziare
(prompt) Say, formally, to a woman: I thank you. || La ringrazio.
(prompt) Say to a friend: thank you for everything. Use {ringraziare}. || Ti ringrazio di tutto.
(prompt) Say: we thanked the hosts. || Abbiamo ringraziato i padroni di casa.
(prompt) Tell a child: thank your grandmother! || Ringrazia la nonna!

(section) 8 — scusarsi
(narrator) Verb eight. To apologize: {scusarsi}.
(repeat x3 slow) scusarsi
(prompt) Say: I apologize for the delay. || Mi scuso per il ritardo.
(prompt) Say: he didn't even apologize. || Non si è nemmeno scusato.
(prompt) Say to two friends: you should apologize. || Dovreste scusarvi.
(prompt) Say: I want him to apologize. || Voglio che si scusi.

(section) Review 1
(narrator) Quick review.
(prompt) Say: she accepted the job offer. || Ha accettato l'offerta di lavoro.
(prompt) Say: he turned down the invitation. || Ha rifiutato l'invito.
(prompt) Say: I admit I was wrong. || Ammetto di aver sbagliato.
(prompt) Say: I have nothing to hide. || Non ho niente da nascondere.
(prompt) Say: I found out the truth. || Ho scoperto la verità.
(prompt) Tell a friend: tell me everything! || Raccontami tutto!

(section) 9 — dimostrare
(narrator) Verb nine. To show, to prove: {dimostrare}. With age, it means: to look a certain age.
(repeat x3 slow) dimostrare
(prompt) Say to a friend: you showed a lot of courage. || Hai dimostrato molto coraggio.
(prompt) Ask: how can I prove it? || Come posso dimostrarlo?
(prompt) Say: the data show that the theory is correct. || I dati dimostrano che la teoria è corretta.
(prompt) Say to a friend: you don't look forty! || Non dimostri quarant'anni!

(section) 10 — tradurre
(narrator) Verb ten. To translate: {tradurre}. It is irregular. I translate is {traduco}. Translated is {tradotto}.
(repeat x3 slow) tradurre
(prompt) Ask a friend: can you translate this letter for me? || Mi puoi tradurre questa lettera?
(prompt) Say: I translated the article into English. || Ho tradotto l'articolo in inglese.
(prompt) Say: she translates from German. || Traduce dal tedesco.
(prompt) Say: they will translate the book next year. || Tradurranno il libro l'anno prossimo.

(section) 11 — riconoscere
(narrator) Verb eleven. To recognize, or to acknowledge: {riconoscere}. The past participle is {riconosciuto}.
(repeat x3 slow) riconoscere
(prompt) Say to a friend: I didn't recognize you! || Non ti ho riconosciuto!
(prompt) Ask a friend: do you recognize this song? || Riconosci questa canzone?
(prompt) Say: we must acknowledge that he was right. || Dobbiamo riconoscere che aveva ragione.
(prompt) Say to a friend: with the beard, nobody will recognize you. || Con la barba, nessuno ti riconoscerà.

(section) 12 — confrontare
(narrator) Verb twelve. To compare: {confrontare}.
(repeat x3 slow) confrontare
(prompt) Say: let's compare the prices. || Confrontiamo i prezzi.
(prompt) Say: I compared the two offers. || Ho confrontato le due offerte.
(prompt) Tell a friend: don't compare yourself to others. || Non confrontarti con gli altri.
(prompt) Say: they compared the results with the simulation. || Hanno confrontato i risultati con la simulazione.

(section) 13 — pretendere
(narrator) Verb thirteen. To demand, to expect too much: {pretendere}. Careful! It does not mean "to pretend". To pretend is {fingere}.
(repeat x3 slow) pretendere
(prompt) Say: my boss demands too much. || Il mio capo pretende troppo.
(prompt) Ask two friends: what do you expect from me? || Che cosa pretendete da me?
(prompt) Say: he demanded to be paid immediately. || Ha preteso di essere pagato subito.
(prompt) Say to a friend: I don't expect you to understand. || Non pretendo che tu capisca.

(section) 14 — sopportare
(narrator) Verb fourteen. To stand, to put up with: {sopportare}. Careful again! It does not mean "to support". To support is {sostenere}.
(repeat x3 slow) sopportare
(prompt) Say: I can't stand the heat. || Non sopporto il caldo.
(prompt) Ask a friend: how do you put up with him? || Come fai a sopportarlo?
(prompt) Say: she put up with a lot. || Ha sopportato molto.
(prompt) Say: they will never put up with this situation. || Non sopporteranno mai questa situazione.

(section) 15 — realizzare
(narrator) Verb fifteen. To make real, to carry out, to build: {realizzare}. To realize, in the sense of understand, is usually {rendersi conto}.
(repeat x3 slow) realizzare
(prompt) Say: I fulfilled my dream. || Ho realizzato il mio sogno.
(prompt) Say: we are carrying out an important project. || Stiamo realizzando un progetto importante.
(prompt) Say: the company will build a new bridge. || L'azienda realizzerà un nuovo ponte.
(prompt) Say to a friend: I hope you make your dreams come true. || Spero che tu realizzi i tuoi sogni.

(section) 16 — aggiungere
(narrator) Verb sixteen. To add: {aggiungere}. The past participle is {aggiunto}.
(repeat x3 slow) aggiungere
(prompt) Tell a friend: add a little salt. || Aggiungi un po' di sale.
(prompt) Say: I added my name to the list. || Ho aggiunto il mio nome alla lista.
(prompt) Ask, formally: do you want to add anything? || Vuole aggiungere qualcosa?
(prompt) Say: we will add two more chairs. || Aggiungeremo altre due sedie.

(section) Review 2
(narrator) Quick review.
(prompt) Say: I apologize for the delay. || Mi scuso per il ritardo.
(prompt) Say to a friend: you don't look forty! || Non dimostri quarant'anni!
(prompt) Say: I translated the article into English. || Ho tradotto l'articolo in inglese.
(prompt) Say to a friend: I didn't recognize you! || Non ti ho riconosciuto!
(prompt) Say: my boss demands too much. || Il mio capo pretende troppo.
(prompt) Say: I can't stand the heat. || Non sopporto il caldo.
(prompt) Say: I fulfilled my dream. || Ho realizzato il mio sogno.

(section) 17 — togliere
(narrator) Verb seventeen. To take off, to remove: {togliere}. It is irregular. I remove is {tolgo}. Removed is {tolto}.
(repeat x3 slow) togliere
(prompt) Tell a friend: take off your shoes. || Togliti le scarpe.
(prompt) Say: I removed the photo from my profile. || Ho tolto la foto dal profilo.
(prompt) Ask the waiter, formally: can you leave out the onion? Use {togliere}. || Può togliere la cipolla?
(prompt) Say: they took away his licence. || Gli hanno tolto la patente.

(section) 18 — accendere
(narrator) Verb eighteen. To turn on, to light: {accendere}. The past participle is {acceso}.
(repeat x3 slow) accendere
(prompt) Tell a friend: turn on the light, please. || Accendi la luce, per favore.
(prompt) Ask: who turned on the heating? || Chi ha acceso il riscaldamento?
(prompt) Ask, formally: do you mind if I turn on the TV? || Le dispiace se accendo la TV?
(prompt) Say: in the evening we used to light the fireplace. || La sera accendevamo il camino.

(section) 19 — spostare
(narrator) Verb nineteen. To move something, or to reschedule: {spostare}. To move yourself is {spostarsi}.
(repeat x3 slow) spostare
(prompt) Ask: can we move the meeting? || Possiamo spostare la riunione?
(prompt) Say: I moved the car. || Ho spostato la macchina.
(prompt) Tell a friend: move over! || Spostati!
(prompt) Say to two friends: if you move the table, there will be more space. || Se spostate il tavolo, ci sarà più spazio.

(section) 20 — rimandare
(narrator) Verb twenty. To postpone, to put off: {rimandare}.
(repeat x3 slow) rimandare
(narrator) A famous proverb. Listen.
(repeat x2 slow) Non rimandare a domani quello che puoi fare oggi.
(prompt) Say the proverb: don't put off until tomorrow what you can do today. || Non rimandare a domani quello che puoi fare oggi.
(prompt) Say: the match was postponed. || La partita è stata rimandata.
(prompt) Say: we will postpone the trip. || Rimanderemo il viaggio.
(prompt) Say: I always used to put off the dentist. || Rimandavo sempre il dentista.

(section) 21 — riempire
(narrator) Verb twenty-one. To fill: {riempire}. I fill is {riempio}.
(repeat x3 slow) riempire
(prompt) Tell a friend: fill the glass. || Riempi il bicchiere.
(prompt) Say: I filled in the form. || Ho riempito il modulo.
(prompt) Say: the square will fill with people. || La piazza si riempirà di gente.
(prompt) Say to a friend: your news fills me with joy. || La tua notizia mi riempie di gioia.

(section) 22 — svolgere
(narrator) Verb twenty-two. To carry out, to do a task: {svolgere}. With {si}, it means: to take place. The past participle is {svolto}.
(repeat x3 slow) svolgere
(prompt) Say: I carried out my research in Bologna. || Ho svolto la mia ricerca a Bologna.
(prompt) Ask, formally: what job do you do? Use {svolgere}. || Che lavoro svolge?
(prompt) Say: the meeting will take place in Rome. || La riunione si svolgerà a Roma.
(prompt) Say: the story takes place in Sicily. || La storia si svolge in Sicilia.

(section) 23 — tenere
(narrator) Verb twenty-three. To hold, to keep: {tenere}. I hold is {tengo}. With {ci}, {tenerci} means: to care a lot about something.
(repeat x3 slow) tenere
(prompt) Ask a friend: hold my bag for me. Use the imperative. || Tienimi la borsa.
(prompt) Say: I really care about it. || Ci tengo molto.
(prompt) Say, formally: keep the change. || Tenga il resto.
(prompt) Say: we kept in touch. || Ci siamo tenuti in contatto.

(section) 24 — coinvolgere
(narrator) Verb twenty-four. To involve: {coinvolgere}. The past participle is {coinvolto}.
(repeat x3 slow) coinvolgere
(prompt) Say: I wasn't involved. || Non sono stato coinvolto.
(prompt) Say: we want to involve the students. || Vogliamo coinvolgere gli studenti.
(prompt) Say: the project involves three universities. || Il progetto coinvolge tre università.
(prompt) Tell a friend: don't involve me in your problems. || Non coinvolgermi nei tuoi problemi.

(section) 25 — insistere
(narrator) Verb twenty-five. To insist: {insistere}. The past participle is {insistito}.
(repeat x3 slow) insistere
(prompt) Say, formally: if you insist... || Se insiste...
(prompt) Say: he insisted on paying. || Ha insistito per pagare.
(prompt) Tell a friend: don't insist! || Non insistere!
(prompt) Say: if they keep saying no, we will insist. || Se continuano a dire di no, insisteremo.

(section) Final review — verbs 1 to 25
(narrator) Final review. Answer quickly.
(prompt) Say: I hope they accept our proposal. || Spero che accettino la nostra proposta.
(prompt) Say: if they offered it to me, I would refuse. || Se me lo offrissero, rifiuterei.
(prompt) Say: they were admitted to university. || Sono stati ammessi all'università.
(prompt) Say: I think she is hiding something. || Credo che nasconda qualcosa.
(prompt) Say to a friend: one day you will find out that I was right. || Un giorno scoprirai che avevo ragione.
(prompt) Say: my grandfather used to tell a lot of stories. || Mio nonno raccontava molte storie.
(prompt) Say, formally, to a woman: I thank you. || La ringrazio.
(prompt) Say: I want him to apologize. || Voglio che si scusi.
(prompt) Say: the data show that the theory is correct. || I dati dimostrano che la teoria è corretta.
(prompt) Say: they will translate the book next year. || Tradurranno il libro l'anno prossimo.
(prompt) Say: we must acknowledge that he was right. || Dobbiamo riconoscere che aveva ragione.
(prompt) Say: they compared the results with the simulation. || Hanno confrontato i risultati con la simulazione.
(prompt) Say to a friend: I don't expect you to understand. || Non pretendo che tu capisca.
(prompt) Ask a friend: how do you put up with him? || Come fai a sopportarlo?
(prompt) Say to a friend: I hope you make your dreams come true. || Spero che tu realizzi i tuoi sogni.
(prompt) Tell a friend: add a little salt. || Aggiungi un po' di sale.
(prompt) Say: they took away his licence. || Gli hanno tolto la patente.
(prompt) Ask: who turned on the heating? || Chi ha acceso il riscaldamento?
(prompt) Ask: can we move the meeting? || Possiamo spostare la riunione?
(prompt) Say: don't put off until tomorrow what you can do today. || Non rimandare a domani quello che puoi fare oggi.
(prompt) Say: I filled in the form. || Ho riempito il modulo.
(prompt) Say: the meeting will take place in Rome. || La riunione si svolgerà a Roma.
(prompt) Say: I really care about it. || Ci tengo molto.
(prompt) Say: the project involves three universities. || Il progetto coinvolge tre università.
(prompt) Say: he insisted on paying. || Ha insistito per pagare.
(narrator) This is the end of lesson nine. You now know seventy-five useful verbs. {Complimenti, e a presto!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
